# M6 — first training: T1 random report + T3a PPO on MAP01

Settings: **Accelerator: GPU T4 x2**, **Internet: on**. Run with **Save & Run All (Commit)**, so a browser disconnect cannot kill it. Expect **about 1–1.5 hours**.

1. Tests that need torch / sb3-contrib (PPO smoke: train, checkpoint, resume), plus the report and runner tests.
2. **T1 random on MAP03, 200k steps** → `results/t1_random_map03_seed0/MAP03/report.html`. M6 needs at least one out-of-bounds finding with a working demo.
3. **T3a (PPO, alpha = 1) on MAP01, 1M steps, seeds 0 and 1**, one per T4, in parallel. M6 needs the policy to reach MAP01's exit in **> 80% of eval episodes**.
4. A summary of both acceptance checks.

When it finishes: open the **Output** tab, download `m6_results.zip`, and paste the output of the last two cells back to Claude.

In [ ]:
import os
# Must be set before anything imports torch or vizdoom (PLAN: Kaggle operating rules 6).
os.environ["SDL_VIDEODRIVER"] = "dummy"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

REPO_URL = "https://github.com/Satyajit-147/VizDoom.git"
BRANCH = "main"
WORKDIR = "/kaggle/working/rl-qa-tester"
OUT = "/kaggle/working"

In [ ]:
import subprocess

url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{token}@")
except Exception:
    pass  # public repo, or no secret attached

subprocess.run(["rm", "-rf", WORKDIR], check=True)
subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, WORKDIR], check=True)
os.chdir(WORKDIR)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q -e ".[train,dev]"

In [ ]:
import sys, torch, vizdoom, gymnasium, stable_baselines3, sb3_contrib
print("python", sys.version.split()[0], "| torch", torch.__version__, "| cuda", torch.cuda.is_available(), torch.cuda.device_count())
print("vizdoom", vizdoom.__version__, "| gymnasium", gymnasium.__version__, "| sb3", stable_baselines3.__version__, "| sb3-contrib", sb3_contrib.__version__)

In [ ]:
!python -m pytest tests/test_ppo_smoke.py tests/test_report.py tests/test_runner.py tests/test_env_contract.py -q

In [ ]:
# T1: uniform random on MAP03 for 200k steps, 4 workers. No GPU used.
!python scripts/train.py --config configs/t1_random_map03.yaml --runs-dir {OUT}/runs --results-dir {OUT}/results

In [ ]:
%%bash
# T3a: two seeds at once, one per T4, 2 env workers each (PLAN: Launching two jobs in one Kaggle cell).
cd /kaggle/working/rl-qa-tester
export SDL_VIDEODRIVER=dummy OMP_NUM_THREADS=1 MKL_NUM_THREADS=1
mkdir -p /kaggle/working/logs
for SEED in 0 1; do
  CUDA_VISIBLE_DEVICES=$SEED nohup python scripts/train.py --config configs/t3a_map01.yaml --seed $SEED \
      --runs-dir /kaggle/working/runs --results-dir /kaggle/working/results \
      > /kaggle/working/logs/t3a_seed$SEED.log 2>&1 &
done
wait
tail -n 25 /kaggle/working/logs/t3a_seed0.log /kaggle/working/logs/t3a_seed1.log

In [ ]:
import json, pathlib
out = pathlib.Path(OUT)
print("=== T1 random, MAP03 ===")
rep = json.loads((out / "results/t1_random_map03_seed0/MAP03/report.json").read_text())
print(f"{rep['env_steps']:,} steps in {rep['wall_clock_s']:.0f}s, {rep['raw_findings']} raw findings, coverage {rep['coverage']}")
for f in rep["findings"]:
    print(f"  {f['id']:18s} sev {f['severity']:.2f} x{f['cluster_size']:<4d} demo={f['repro']['demo']}  {f['summary']}")
oob = [f for f in rep["findings"] if f["oracle"] == "out_of_bounds" and f["repro"]["demo"]]
print("M6 check 1 (out-of-bounds finding with a demo):", "PASS" if oob else "FAIL")
print()
print("=== T3a PPO, MAP01 ===")
for seed in (0, 1):
    run = out / f"runs/t3a_map01_seed{seed}"
    evals = [json.loads(l) for l in (run / "eval.jsonl").read_text().splitlines()] if (run / "eval.jsonl").exists() else []
    curve = ", ".join(f"{e['step'] // 1000}k:{e['maps']['MAP01']['goal_rate']:.2f}" for e in evals)
    final = evals[-1]["maps"]["MAP01"] if evals else None
    print(f"seed {seed}: goal rate by step  {curve}")
    if final:
        print(f"  final: goal {final['goal_rate']:.2f} over {final['episodes']} episodes, {final['mean_steps']:.0f} steps, return {final['mean_return']:.2f}",
              "-> M6 check 2:", "PASS" if final["goal_rate"] > 0.8 else "FAIL")

In [ ]:
!cd /kaggle/working && zip -qr m6_results.zip results logs runs/*/metrics.jsonl runs/*/eval.jsonl runs/*/manifest.json runs/*/episodes.jsonl && ls -la m6_results.zip